# 🧪 Lab 02 — Lost in Projection: `GEOMETRY` vs `GEOGRAPHY` 📐🌎

Spark 4.2 can recognize spatial data. Great. Now we make the map dangerous.

This lab asks:

> **What is the real difference between `GEOMETRY` and `GEOGRAPHY`, and which coordinate mistakes can Spark actually catch?**

We use stock **PySpark 4.2.0** only — no Sedona, Shapely, or GeoPandas.

### 🎯 Mission objectives

We will prove that:

- the **same WKB point** can become either `GEOMETRY` or `GEOGRAPHY`;
- `GEOMETRY` is Cartesian / graph-paper space;
- `GEOGRAPHY` is longitude/latitude space and enforces geographic bounds;
- `GEOMETRY(3857)` is valid while `GEOGRAPHY(3857)` is not;
- `GEOMETRY(0)` means an unspecified Cartesian reference;
- `GEOMETRY(ANY)` can honestly represent mixed SRIDs without transforming them;
- Spark can reject structural spatial nonsense;
- Spark cannot detect a perfectly valid but semantically swapped Madrid coordinate.

And yes:

```text
where_the_hell_is_madrid()
```

is now a scientific instrument.

> **Evidence boundary:** this notebook empirically tests Spark 4.2's type construction, SRID rules, geographic coordinate validation, mixed-SRID widening, and axis-order failure mode. Spark documents `GEOGRAPHY` edges as using spherical interpolation, but stock Spark 4.2 does not expose a native distance/predicate operation here that lets this lab isolate that behavior directly. We therefore treat spherical interpolation as a **documented type semantic**, not as an experimentally proven result of this notebook.

## 0 — Pre-flight checks 🛰️

Target environment:

```text
PySpark  4.2.0
Spark    4.2.0
Java     17+
```

The Windows-safe bootstrap from Lab 01 is included so a missing `PYSPARK_SUBMIT_ARGS` does not kill the Java gateway before take-off.

In [9]:
import sys, json, warnings, struct, math


warnings.filterwarnings(
    "ignore",
    message=r"PySpark does not yet fully support pandas >= 3\.0\.0.*",
    category=FutureWarning,
)

import pyspark
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import GeometryType, GeographyType

active = SparkSession.getActiveSession()
if active is not None:
    active.stop()

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("lab-02-geometry-vs-geography-lost-in-projection")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")

java_version = spark.sparkContext._jvm.java.lang.System.getProperty("java.version")
fingerprint = {
    "python": sys.version.split()[0],
    "pyspark": pyspark.__version__,
    "spark": spark.version,
    "java": java_version,
    "geospatial_enabled": spark.conf.get("spark.sql.geospatial.enabled"),
}

print("🚀 Runtime fingerprint")
print(json.dumps(fingerprint, indent=2))

assert pyspark.__version__ == "4.2.0"
assert spark.version == "4.2.0"
assert int(java_version.split(".")[0]) >= 17
assert fingerprint["geospatial_enabled"].lower() == "true"

print("\n✅ Navigation computer online.")

🚀 Runtime fingerprint
{
  "python": "3.14.0",
  "pyspark": "4.2.0",
  "spark": "4.2.0",
  "java": "17.0.19",
  "geospatial_enabled": "true"
}

✅ Navigation computer online.


# 1 — Same Coordinates, Two Universes 📐🌎

Start with the classic graph-paper points:

```text
A = POINT(0 0)
B = POINT(3 4)
```

Cartesian maths says the distance is `5`.

Five **what**? Metres? Feet? Map units? Bananas? 🍌

Now we take the **same WKB bytes** and ask Spark to interpret them as:

```text
GEOMETRY  → Cartesian / planar contract
GEOGRAPHY → geographic longitude/latitude contract
```

Stock Spark 4.2 has no native `ST_Distance`, so the `5` below is deliberately computed with Python's ordinary Euclidean maths on the source numbers. It **illustrates Cartesian graph-paper semantics**; it is not a Spark geospatial result.

In [10]:
def wkb_point(x, y):
    return struct.pack("<BIdd", 1, 1, float(x), float(y))

points = spark.createDataFrame(
    [
        ("A", 0.0, 0.0, wkb_point(0, 0)),
        ("B", 3.0, 4.0, wkb_point(3, 4)),
    ],
    ["name", "x", "y", "wkb"],
)

dual = (
    points
    .withColumn("as_geometry", F.st_geomfromwkb("wkb"))
    .withColumn("as_geography", F.st_geogfromwkb("wkb"))
)

telemetry = dual.select(
    "name", "x", "y",
    F.expr("typeof(as_geometry)").alias("geometry_type"),
    F.st_srid("as_geometry").alias("geometry_srid"),
    F.expr("typeof(as_geography)").alias("geography_type"),
    F.st_srid("as_geography").alias("geography_srid"),
)

print("📐 SAME WKB, TWO CONTRACTS")
telemetry.show(truncate=False)

euclidean_distance = math.hypot(3.0, 4.0)
print(f"🍌 Graph-paper arithmetic: distance = {euclidean_distance}")
print("   Unit? The coordinate system has to tell us.")

rows = telemetry.collect()
assert all(r.geometry_type.lower() == "geometry(0)" for r in rows)
assert all(r.geometry_srid == 0 for r in rows)
assert all(r.geography_type.lower() == "geography(4326)" for r in rows)
assert all(r.geography_srid == 4326 for r in rows)
assert euclidean_distance == 5.0

lab_results = {
    "euclidean_distance": euclidean_distance,
    "geometry_srid": rows[0].geometry_srid,
    "geography_srid": rows[0].geography_srid,
}

📐 SAME WKB, TWO CONTRACTS
+----+---+---+-------------+-------------+---------------+--------------+
|name|x  |y  |geometry_type|geometry_srid|geography_type |geography_srid|
+----+---+---+-------------+-------------+---------------+--------------+
|A   |0.0|0.0|geometry(0)  |0            |geography(4326)|4326          |
|B   |3.0|4.0|geometry(0)  |0            |geography(4326)|4326          |
+----+---+---+-------------+-------------+---------------+--------------+

🍌 Graph-paper arithmetic: distance = 5.0
   Unit? The coordinate system has to tell us.


### First result

The WKB did not change. The **contract** did.

```text
ST_GeomFromWKB(wkb)
→ GEOMETRY(0)

ST_GeogFromWKB(wkb)
→ GEOGRAPHY(4326)
```

`GEOMETRY(0)` means Cartesian Geometry with an unspecified coordinate reference.

`GEOGRAPHY(4326)` means geographic longitude/latitude coordinates.

Same bytes. Very different promise.

# 2 — Graph Paper Accepts Things Earth Refuses 🗺️💥

Consider:

```text
POINT(1000000 5000000)
```

In projected or engineering coordinates, those numbers may be perfectly reasonable.

As longitude/latitude they are hilariously impossible.

Spark's Geography parser requires:

```text
longitude ∈ [-180, 180]
latitude  ∈ [ -90,  90]
```

So we feed the **same WKB** to Geometry and Geography.

In [11]:
wild = spark.createDataFrame(
    [(wkb_point(1_000_000, 5_000_000),)],
    ["wkb"],
)

geom = wild.select(F.st_geomfromwkb("wkb", 3857).alias("geom"))
geom_row = geom.select(
    F.expr("typeof(geom)").alias("spark_type"),
    F.st_srid("geom").alias("srid"),
).first()

print("📐 GEOMETRY")
print(f"  ├─ type : {geom_row.spark_type}")
print(f"  └─ SRID : {geom_row.srid}")

assert geom_row.spark_type.lower() == "geometry(3857)"
assert geom_row.srid == 3857

def compact_error(exc):
    lines = [x.strip() for x in str(exc).splitlines() if x.strip()]
    return lines[0] if lines else repr(exc)

geog_bounds_error = None
spark.sparkContext.setLogLevel("OFF")
try:
    wild.select(F.st_geogfromwkb("wkb").alias("geog")).collect()
except Exception as exc:
    geog_bounds_error = exc
finally:
    spark.sparkContext.setLogLevel("ERROR")

print("\n🌎 GEOGRAPHY")
print(f"  ├─ accepted : {geog_bounds_error is None}")
if geog_bounds_error:
    print(f"  └─ error    : {compact_error(geog_bounds_error)}")

assert geog_bounds_error is not None

lab_results["geometry_accepts_projected_coords"] = True
lab_results["geography_rejects_out_of_bounds"] = True
lab_results["geography_bounds_error"] = type(geog_bounds_error).__name__

📐 GEOMETRY
  ├─ type : geometry(3857)
  └─ SRID : 3857

🌎 GEOGRAPHY
  ├─ accepted : False
  └─ error    : [WKB_PARSE_ERROR] Error parsing WKB: Invalid coordinate value found at position 5 SQLSTATE: 22023


# 3 — `3857`: Geometry Yes, Geography No 🛰️

`3857` is Web Mercator, a projected Cartesian reference used heavily by web maps.

That makes it valid for `GEOMETRY`.

But `GEOGRAPHY` only accepts **geographic** SRIDs.

Let's test the actual PySpark type constructors.

In [12]:
geometry_0 = GeometryType(0)
geometry_3857 = GeometryType(3857)
geometry_any = GeometryType("ANY")
geography_4326 = GeographyType(4326)
geography_any = GeographyType("ANY")

geography_3857_error = None
try:
    GeographyType(3857)
except Exception as exc:
    geography_3857_error = exc

print("🧭 Spatial type probe")
print(f"  ├─ {geometry_0.simpleString()}")
print(f"  ├─ {geometry_3857.simpleString()}")
print(f"  ├─ {geometry_any.simpleString()}")
print(f"  ├─ {geography_4326.simpleString()}")
print(f"  ├─ {geography_any.simpleString()}")
print(f"  └─ GeographyType(3857) accepted? {geography_3857_error is None}")

if geography_3857_error:
    print(f"     error: {compact_error(geography_3857_error)}")

assert geometry_3857.simpleString().lower() == "geometry(3857)"
assert geography_4326.simpleString().lower() == "geography(4326)"
assert "any" in geometry_any.simpleString().lower()
assert "any" in geography_any.simpleString().lower()
assert geography_3857_error is not None

lab_results["geography_3857_rejected"] = True
lab_results["geography_3857_error"] = type(geography_3857_error).__name__

🧭 Spatial type probe
  ├─ geometry(0)
  ├─ geometry(3857)
  ├─ geometry(any)
  ├─ geography(4326)
  ├─ geography(any)
  └─ GeographyType(3857) accepted? False
     error: [ST_INVALID_SRID_VALUE] Invalid or unsupported SRID (spatial reference identifier) value: 3857.


# 4 — `GEOMETRY(ANY)`: The Spatial Multiverse 🌌

`GEOMETRY(ANY)` allows rows to carry different valid SRIDs.

It does **not** mean Spark transformed them into one CRS.

We create one native Geometry with SRID `4326` and another with `3857`, then union them. Spark should widen the common result to a mixed-SRID Geometry type.

In [13]:
g4326_df = (
    spark.createDataFrame([(1, wkb_point(-3.7038, 40.4168))], ["id", "wkb"])
    .select("id", F.st_geomfromwkb("wkb", 4326).alias("geom"))
)

g3857_df = (
    spark.createDataFrame([(2, wkb_point(-412305.13, 4926696.67))], ["id", "wkb"])
    .select("id", F.st_geomfromwkb("wkb", 3857).alias("geom"))
)

mixed = g4326_df.unionByName(g3857_df)

print("🌌 MIXED-SRID UNION")
mixed.printSchema()

mixed_telemetry = mixed.select(
    "id",
    F.expr("typeof(geom)").alias("column_type"),
    F.st_srid("geom").alias("value_srid"),
).orderBy("id")

mixed_telemetry.show(truncate=False)

mixed_rows = mixed_telemetry.collect()
mixed_type = mixed_rows[0].column_type.lower()
mixed_srids = [r.value_srid for r in mixed_rows]

assert mixed_type == "geometry(any)"
assert mixed_srids == [4326, 3857]

print("📡 Translation: Spark preserved the disagreement. It did not reproject anything.")

lab_results["mixed_type"] = mixed_type
lab_results["mixed_srids"] = mixed_srids

🌌 MIXED-SRID UNION
root
 |-- id: long (nullable = true)
 |-- geom: geometry(any) (nullable = true)

+---+-------------+----------+
|id |column_type  |value_srid|
+---+-------------+----------+
|1  |geometry(any)|4326      |
|2  |geometry(any)|3857      |
+---+-------------+----------+

📡 Translation: Spark preserved the disagreement. It did not reproject anything.


# 5 — Longitude / Latitude Tries to Murder Us 🤡🌍

Madrid is approximately:

```text
longitude = -3.7038
latitude  = 40.4168
```

Spatial coordinates normally follow `X, Y`, so geographic data is normally written:

```text
longitude, latitude
```

Correct:

```text
POINT(-3.7038 40.4168)
```

Swapped:

```text
POINT(40.4168 -3.7038)
```

And here comes the nasty part:

```text
40.4168  is a legal longitude
-3.7038  is a legal latitude
```

Both values are structurally valid.

Let's see whether Spark can read our mind.

The thousands-of-kilometres separation printed later is also a **pure-Python Haversine diagnostic**. Spark's experimental contribution is the more important one: **both coordinate pairs are accepted as valid `GEOGRAPHY(4326)` values**.

In [14]:
madrid = spark.createDataFrame(
    [
        ("correct_madrid", -3.7038, 40.4168, wkb_point(-3.7038, 40.4168)),
        ("swapped_madrid", 40.4168, -3.7038, wkb_point(40.4168, -3.7038)),
    ],
    ["label", "x_longitude", "y_latitude", "wkb"],
).withColumn(
    "geog",
    F.st_geogfromwkb("wkb"),
)

madrid_telemetry = madrid.select(
    "label", "x_longitude", "y_latitude",
    F.expr("typeof(geog)").alias("spark_type"),
    F.st_srid("geog").alias("srid"),
)

print("🤡 BOTH PASS GEOGRAPHY VALIDATION")
madrid_telemetry.show(truncate=False)

accepted = madrid_telemetry.collect()
assert len(accepted) == 2
assert all(r.spark_type.lower() == "geography(4326)" for r in accepted)
assert all(r.srid == 4326 for r in accepted)

def haversine_km(lon1, lat1, lon2, lat2):
    # Pure-Python diagnostic. NOT a Spark ST_Distance call.
    radius = 6371.0088
    lon1, lat1, lon2, lat2 = map(math.radians, [lon1, lat1, lon2, lat2])
    dlon, dlat = lon2 - lon1, lat2 - lat1
    a = (
        math.sin(dlat / 2) ** 2
        + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2) ** 2
    )
    return 2 * radius * math.asin(math.sqrt(a))

swap_distance_km = haversine_km(
    -3.7038, 40.4168,
    40.4168, -3.7038,
)

def where_the_hell_is_madrid():
    print("🛰️ where_the_hell_is_madrid()")
    print("  ├─ intended : lon=-3.7038, lat=40.4168")
    print("  ├─ swapped  : lon=40.4168, lat=-3.7038")
    print("  ├─ intended accepted by Spark : ✅")
    print("  ├─ swapped accepted by Spark  : ✅")
    print(f"  └─ pure-Python great-circle diagnostic: {swap_distance_km:,.0f} km apart")
    print("\n💥 Semantically wrong. Computationally perfect.")

where_the_hell_is_madrid()

lab_results["madrid_swap_distance_km"] = swap_distance_km
lab_results["both_madrids_valid"] = True

🤡 BOTH PASS GEOGRAPHY VALIDATION
+--------------+-----------+----------+---------------+----+
|label         |x_longitude|y_latitude|spark_type     |srid|
+--------------+-----------+----------+---------------+----+
|correct_madrid|-3.7038    |40.4168   |geography(4326)|4326|
|swapped_madrid|40.4168    |-3.7038   |geography(4326)|4326|
+--------------+-----------+----------+---------------+----+

🛰️ where_the_hell_is_madrid()
  ├─ intended : lon=-3.7038, lat=40.4168
  ├─ swapped  : lon=40.4168, lat=-3.7038
  ├─ intended accepted by Spark : ✅
  ├─ swapped accepted by Spark  : ✅
  └─ pure-Python great-circle diagnostic: 6,646 km apart

💥 Semantically wrong. Computationally perfect.


# 6 — What Spark Can Know vs What Only You Can Know 🧠

Spark can reject machine-readable contradictions:

```text
GEOGRAPHY with impossible longitude/latitude
GEOGRAPHY with a projected SRID such as 3857
```

It can also represent mixed SRIDs honestly as `GEOMETRY(ANY)`.

But Spark cannot infer that:

```text
POINT(40.4168 -3.7038)
```

was supposed to mean Madrid.

That is the difference between:

```text
valid
```

and:

```text
correct
```

A type system can validate the first.

Domain knowledge has to establish the second.

# 📊 Post-Lab Analysis — Build the Verdict From the Evidence

The next cell generates the conclusion from the values observed during this execution.

If Spark behaves differently, an assertion above stops the mission instead of letting the notebook print a victory speech from another universe.

In [15]:
from IPython.display import Markdown, display

analysis = f"""
# 📊 Post-Lab Analysis: Flat Space, Round Earth, and One Missing Madrid

The same WKB points became `GEOMETRY({lab_results['geometry_srid']})` and `GEOGRAPHY({lab_results['geography_srid']})`. Using ordinary **Python Euclidean maths** as a graph-paper illustration, `(0,0)` to `(3,4)` produced **{lab_results['euclidean_distance']} coordinate units**. That number is not a Spark `ST_Distance` result, and its physical unit cannot be inferred from the coordinates alone.

### 1. Geography Is Not Geometry With a Fancier Name

Projected-looking coordinates were accepted as Geometry:

**{lab_results['geometry_accepts_projected_coords']}**

The same payload was rejected as Geography because its coordinates were outside legal longitude/latitude bounds:

**{lab_results['geography_rejects_out_of_bounds']}**

`GeographyType(3857)` was also rejected:

**{lab_results['geography_3857_rejected']}**

Spark is therefore empirically enforcing a real difference between Cartesian and geographic spatial contracts at the type/parsing boundary.

### 2. `ANY` Represents Disagreement — It Does Not Fix It

Unioning `GEOMETRY(4326)` with `GEOMETRY(3857)` produced:

**`{lab_results['mixed_type']}`**

while the individual values still carried:

```text
SRIDs = {lab_results['mixed_srids']}
```

Spark preserved the mismatch. It did not reproject either row.

### 3. The Most Dangerous Error Was Perfectly Valid

Both Madrid coordinate pairs were accepted as `GEOGRAPHY(4326)`:

```text
correct : (-3.7038, 40.4168)
swapped : (40.4168, -3.7038)
```

A deliberately separate **pure-Python Haversine diagnostic** puts those two Spark-valid positions roughly:

**{lab_results['madrid_swap_distance_km']:,.0f} km apart**

Spark did nothing wrong. Every number was inside the legal geographic range.

The bug lived entirely in our intent.

> ## 🚀 Mission Verdict
> **`GEOMETRY` and `GEOGRAPHY` are not cosmetic aliases.** This lab proves different type/SRID/validation behavior: Geometry accepts Cartesian/projected contracts, while Geography enforces geographic SRIDs and longitude/latitude validity.
>
> Spark 4.2 can catch structural spatial lies. But when the lie is perfectly valid — like swapping Madrid's longitude and latitude — Spark validates it and sends the city thousands of kilometres away.
>
> **Spatial types reduce the number of ways we can lie to Spark. They do not stop us from lying very convincingly.**

### Evidence note

This lab directly proves Spark's **type, SRID, validation, mixed-SRID, and axis-order behavior**. The section's statement that `GEOGRAPHY` uses **spherical edge interpolation** comes from Spark's documented type semantics; this notebook does not claim to measure that interpolation.
"""

display(Markdown(analysis))


# 📊 Post-Lab Analysis: Flat Space, Round Earth, and One Missing Madrid

The same WKB points became `GEOMETRY(0)` and `GEOGRAPHY(4326)`. Using ordinary **Python Euclidean maths** as a graph-paper illustration, `(0,0)` to `(3,4)` produced **5.0 coordinate units**. That number is not a Spark `ST_Distance` result, and its physical unit cannot be inferred from the coordinates alone.

### 1. Geography Is Not Geometry With a Fancier Name

Projected-looking coordinates were accepted as Geometry:

**True**

The same payload was rejected as Geography because its coordinates were outside legal longitude/latitude bounds:

**True**

`GeographyType(3857)` was also rejected:

**True**

Spark is therefore empirically enforcing a real difference between Cartesian and geographic spatial contracts at the type/parsing boundary.

### 2. `ANY` Represents Disagreement — It Does Not Fix It

Unioning `GEOMETRY(4326)` with `GEOMETRY(3857)` produced:

**`geometry(any)`**

while the individual values still carried:

```text
SRIDs = [4326, 3857]
```

Spark preserved the mismatch. It did not reproject either row.

### 3. The Most Dangerous Error Was Perfectly Valid

Both Madrid coordinate pairs were accepted as `GEOGRAPHY(4326)`:

```text
correct : (-3.7038, 40.4168)
swapped : (40.4168, -3.7038)
```

A deliberately separate **pure-Python Haversine diagnostic** puts those two Spark-valid positions roughly:

**6,646 km apart**

Spark did nothing wrong. Every number was inside the legal geographic range.

The bug lived entirely in our intent.

> ## 🚀 Mission Verdict
> **`GEOMETRY` and `GEOGRAPHY` are not cosmetic aliases.** This lab proves different type/SRID/validation behavior: Geometry accepts Cartesian/projected contracts, while Geography enforces geographic SRIDs and longitude/latitude validity.
>
> Spark 4.2 can catch structural spatial lies. But when the lie is perfectly valid — like swapping Madrid's longitude and latitude — Spark validates it and sends the city thousands of kilometres away.
>
> **Spatial types reduce the number of ways we can lie to Spark. They do not stop us from lying very convincingly.**

### Evidence note

This lab directly proves Spark's **type, SRID, validation, mixed-SRID, and axis-order behavior**. The section's statement that `GEOGRAPHY` uses **spherical edge interpolation** comes from Spark's documented type semantics; this notebook does not claim to measure that interpolation.


## ✅ What this lab actually proves

From the executed Spark 4.2 results:

```text
same WKB → GEOMETRY(0) vs GEOGRAPHY(4326)     ✅
projected-looking coordinates accepted as Geometry ✅
out-of-range lon/lat rejected as Geography    ✅
GEOMETRY(3857) valid                           ✅
GEOGRAPHY(3857) rejected                       ✅
mixed 4326 + 3857 → GEOMETRY(ANY)              ✅
both correct and swapped Madrid accepted       ✅
valid coordinates ≠ correct location           ✅
```

And one thing remains deliberately **documentation-backed rather than lab-measured**:

```text
GEOGRAPHY spherical edge interpolation         📚
```

# 🛰️ Mission Handoff

We now know:

```text
GEOMETRY  → Cartesian contract
GEOGRAPHY → geographic contract
SRID 0    → unspecified Cartesian reference
ANY       → mixed valid SRIDs
valid     ≠ correct
```

Next question:

> If the SRID is wrong, can we just call `ST_SetSrid` and fix it?

Oh, sweet summer data engineer.

Next mission: **SRIDs, `ST_SetSrid`, and the Fake Reprojection Crime Scene.** 🚨🛰️

---

## 📚 Primary references

- Apache Spark 4.2 — Geospatial (`GEOMETRY` / `GEOGRAPHY`) types  
  https://spark.apache.org/docs/latest/sql-ref-geospatial-types.html

- Apache Spark 4.2 — SQL data types  
  https://spark.apache.org/docs/latest/sql-ref-datatypes.html

- PySpark 4.2 — geospatial `ST_*` functions  
  https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html

In [16]:
spark.stop()
print("🌍 Spark stopped. Madrid may now be returned to its original coordinates.")

🌍 Spark stopped. Madrid may now be returned to its original coordinates.
